# OS Project FINAL

### Spotify API Implementation

In [152]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from collections import deque
import random
import time

# Set up Spotify API credentials
SPOTIPY_CLIENT_ID = "b7e3d554cb98481a9ed7879d335b5587"
SPOTIPY_CLIENT_SECRET = "dd489cecf99f4063a92c5d793688cb12"

# Authenticate with Spotify
client_credentials_manager = SpotifyClientCredentials(client_id=SPOTIPY_CLIENT_ID, client_secret=SPOTIPY_CLIENT_SECRET)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

# Function to get a list of popular songs
def get_spotify_songs(limit=50):
    results = sp.search(q="Top Hits", limit=limit, type="track")
    songs = [track["name"] + " - " + track["artists"][0]["name"] for track in results["tracks"]["items"]]
    return songs

### Normal Streaming

In [155]:
class NormalStreaming:
    def __init__(self):
        self.requests = 0

    def access_song(self, song):
        self.requests += 1
        # Simulate full fetch time (no cache benefit)
        time.sleep(0.005)

# Simulating Music Streaming
cache_size = 5
normal_streaming = NormalStreaming()

# Get Spotify Songs
spotify_songs = get_spotify_songs(limit=50)
song_requests = [random.choice(spotify_songs) for _ in range(100)]

# Measure Normal Streaming Performance
start_time_normal = time.time()
for song in song_requests:
    normal_streaming.access_song(song)
end_time_normal = time.time()

print(f"🎵 Total Songs Requested: {len(song_requests)}")
print(f"🔄 Normal Streaming Requests: {normal_streaming.requests}")
print(f"⏳ Normal Streaming Time: {end_time_normal - start_time_normal:.5f} sec")

🎵 Total Songs Requested: 100
🔄 Normal Streaming Requests: 100
⏳ Normal Streaming Time: 0.62053 sec


### --------------------------------------

### FIFO (First In First Out) Cache Implementation

In [158]:
class FIFOCache:
    def __init__(self, size):
        self.cache = deque(maxlen=size)
        self.cache_set = set()
        self.cache_hits = 0
        self.cache_misses = 0

    def access_song(self, song):
        if song in self.cache_set:
            self.cache_hits += 1
            # Simulate fast access for cache hit
            time.sleep(0.001)
        else:
            self.cache_misses += 1
            if len(self.cache) >= self.cache.maxlen:
                removed_song = self.cache.popleft()
                self.cache_set.remove(removed_song)
            self.cache.append(song)
            self.cache_set.add(song)
            # Simulate longer access time for cache miss (like fetching from server)
            time.sleep(0.005)

# Simulating Music Streaming
cache_size = 5
fifo_cache = FIFOCache(cache_size)
normal_streaming = NormalStreaming()

# Measure FIFO Performance
start_time_fifo = time.time()
for song in song_requests:
    fifo_cache.access_song(song)
end_time_fifo = time.time()

# Results
print("\n📊 RESULTS:")
print(f"🎵 Total Songs Requested: {len(song_requests)}")
print(f"✅ FIFO Cache Hits: {fifo_cache.cache_hits}")
print(f"🚫 FIFO Cache Misses: {fifo_cache.cache_misses}")
print(f"⏳ FIFO Processing Time: {end_time_fifo - start_time_fifo:.5f} sec")


📊 RESULTS:
🎵 Total Songs Requested: 100
✅ FIFO Cache Hits: 19
🚫 FIFO Cache Misses: 81
⏳ FIFO Processing Time: 0.52123 sec


### --------------------------------------

### LRU (Least Recently Used) Cache Implementation 

In [138]:
from collections import OrderedDict
import random
import time

# LRU Breakdown
class LRUCache:
    def __init__(self, size):
        self.cache = OrderedDict()
        self.max_size = size
        self.cache_hits = 0
        self.cache_misses = 0

    def access_song(self, song):
        if song in self.cache:
            self.cache_hits += 1
            self.cache.move_to_end(song)
        else:
            self.cache_misses += 1
            if len(self.cache) >= self.max_size:
                removed, _ = self.cache.popitem(last=False)
            self.cache[song] = True

# Simulate Streaming with LRU
cache_size = 5
lru_cache = LRUCache(cache_size)

# Simulated playlist
spotify_songs = get_spotify_songs(limit=50)
song_requests = [random.choice(spotify_songs) for _ in range(100)]

start_time = time.time()
for song in song_requests:
    lru_cache.access_song(song)
end_time = time.time()

# Results
print("\n📊 LRU CACHE BREAKDOWN")
print(f"🎵 Total Songs Requested: {len(song_requests)}")
print(f"✅ Cache Hits: {lru_cache.cache_hits}")
print(f"🚫 Cache Misses: {lru_cache.cache_misses}")
print(f"🕒 Processing Time: {end_time - start_time:.5f} sec")
print(f"🎼 Final Cache: {list(lru_cache.cache.keys())}")
print(f"💾 Final Cache Usage: {len(lru_cache.cache)}/{lru_cache.max_size}")



📊 LRU CACHE BREAKDOWN
🎵 Total Songs Requested: 100
✅ Cache Hits: 6
🚫 Cache Misses: 94
🕒 Processing Time: 0.00024 sec
🎼 Final Cache: ['Blue Velvet - Stereo - Bobby Rydell', 'Top Hits - Instrumental Popular Songs 2023', 'Top Hits - Best Songs of 2023 Hit Beats Playlist', 'See You Again - Wiz Khalifa', 'Top Hits - Top TikTok Hits Now']
💾 Final Cache Usage: 5/5


### --------------------------------------

### LFU (Least Frequently Used) Cache Implementation 

In [140]:
from collections import defaultdict
import random
import time

# LFU Breakdown
class LFUCache:
    def __init__(self, size):
        self.cache = {}
        self.freq = defaultdict(int)
        self.max_size = size
        self.cache_hits = 0
        self.cache_misses = 0

    def access_song(self, song):
        if song in self.cache:
            self.freq[song] += 1
            self.cache_hits += 1
        else:
            self.cache_misses += 1
            if len(self.cache) >= self.max_size:
                # Remove the least frequently used song
                least_used = min(self.freq, key=self.freq.get)
                del self.cache[least_used]
                del self.freq[least_used]
            self.cache[song] = True
            self.freq[song] = 1

# Simulate Streaming with LFU
cache_size = 5
lfu_cache = LFUCache(cache_size)

# Simulated playlist
spotify_songs = get_spotify_songs(limit=50)
song_requests = [random.choice(spotify_songs) for _ in range(100)]

start_time = time.time()
for song in song_requests:
    lfu_cache.access_song(song)
end_time = time.time()

# Final Results
print("\n📊 LFU CACHE BREAKDOWN")
print(f"🎵 Total Songs Requested: {len(song_requests)}")
print(f"✅ Cache Hits: {lfu_cache.cache_hits}")
print(f"🚫 Cache Misses: {lfu_cache.cache_misses}")
print(f"🕒 Processing Time: {end_time - start_time:.5f} sec")
print(f"🎼 Final Cache: {list(lfu_cache.cache.keys())}")
print(f"💾 Final Cache Usage: {len(lfu_cache.cache)}/{lfu_cache.max_size}")



📊 LFU CACHE BREAKDOWN
🎵 Total Songs Requested: 100
✅ Cache Hits: 13
🚫 Cache Misses: 87
🕒 Processing Time: 0.00033 sec
🎼 Final Cache: ['Top Hits - Instrumental Popular Songs 2023', 'Top Hits - Best Songs of 2023 Hit Beats Playlist', 'Replay - Iyaz', 'Whistle - Flo Rida', 'Top Hits - Top 10 Hits Today']
💾 Final Cache Usage: 5/5


### --------------------------------------

### MRU (Most Recently Used) Cache Implementation 

In [142]:
import random
import time

# MRU Breakdown
class MRUCache:
    def __init__(self, size):
        self.cache = []
        self.max_size = size
        self.last_accessed = None
        self.cache_hits = 0
        self.cache_misses = 0

    def access_song(self, song):
        if song in self.cache:
            self.cache_hits += 1
            self.last_accessed = song
        else:
            self.cache_misses += 1
            if len(self.cache) >= self.max_size:
                if self.last_accessed and self.last_accessed in self.cache:
                    self.cache.remove(self.last_accessed)
                else:
                    self.cache.pop()  # fallback if last_accessed is missing
            self.cache.append(song)
            self.last_accessed = song

# Simulate Streaming with MRU
cache_size = 5
mru_cache = MRUCache(cache_size)

# Simulated playlist
spotify_songs = get_spotify_songs(limit=50)
song_requests = [random.choice(spotify_songs) for _ in range(100)]

start_time = time.time()
for song in song_requests:
    mru_cache.access_song(song)
end_time = time.time()

# Final Results
print("\n📊 MRU CACHE BREAKDOWN")
print(f"🎵 Total Songs Requested: {len(song_requests)}")
print(f"✅ Cache Hits: {mru_cache.cache_hits}")
print(f"🚫 Cache Misses: {mru_cache.cache_misses}")
print(f"🕒 Processing Time: {end_time - start_time:.5f} sec")
print(f"🎼 Final Cache: {mru_cache.cache}")
print(f"💾 Final Cache Usage: {len(mru_cache.cache)}/{mru_cache.max_size}")



📊 MRU CACHE BREAKDOWN
🎵 Total Songs Requested: 100
✅ Cache Hits: 14
🚫 Cache Misses: 86
🕒 Processing Time: 0.00048 sec
🎼 Final Cache: ['ยิ่งดุยิ่งชอบ - Billkin', 'Top Hits - Top 10 Hits Today', 'Top Hits - Todays Top Hits', 'California Love - Original Version - 2Pac', 'Top Hits - Popular Songs']
💾 Final Cache Usage: 5/5


### --------------------------------------

### Comparing Result

In [145]:
import time
import random

# Function to run a single method
def run_test(method_name, cache_class, song_requests, cache_size=3):
    cache = cache_class(cache_size)
    start = time.time()
    for song in song_requests:
        cache.access_song(song)
    end = time.time()
    duration = end - start
    hits = getattr(cache, 'cache_hits', 0)
    misses = getattr(cache, 'cache_misses', len(song_requests) - hits)
    
    # Individual Result Printout
    print(f"\n--- {method_name} RESULTS ---")
    print(f"✅ Cache Hits: {hits}")
    print(f"🚫 Cache Misses: {misses}")
    print(f"🕒 Time Taken: {duration:.5f} seconds")
    
    return (method_name, hits, misses, duration)

# Dummy Normal Streaming class
class NormalStreaming:
    def __init__(self, size):  # dummy param for compatibility
        self.cache_hits = 0
        self.cache_misses = 0

    def access_song(self, song):
        self.cache_misses += 1
        time.sleep(0.005)

# Simulate Playlist
spotify_songs = get_spotify_songs(limit=50)
song_requests = [random.choice(spotify_songs) for _ in range(100)]

# Display Playlist
print("🎵 Simulated Playlist:")
for i, song in enumerate(song_requests, 1):
    print(f"{i:>2}. {song}")

# Run All Methods
cache_size = 5
results = []

results.append(run_test("FIFO", FIFOCache, song_requests, cache_size))
results.append(run_test("LRU", LRUCache, song_requests, cache_size))
results.append(run_test("LFU", LFUCache, song_requests, cache_size))
results.append(run_test("MRU", MRUCache, song_requests, cache_size))
results.append(run_test("Normal", NormalStreaming, song_requests, cache_size))

# Print Summary Table
print("\n📊 COMPARISON SUMMARY")
print(f"{'Method':<8} | {'Hits':<5} | {'Misses':<7} | {'Time':<10}")
print("-" * 40)
for name, hits, misses, duration in results:
    print(f"{name:<8} | {hits:<5} | {misses:<7} | {duration:.5f} sec")

# Print Best Method
best_method = min(results, key=lambda x: x[3])
print(f"\n🏆 Best Method: {best_method[0]} (Time: {best_method[3]:.5f} sec)")


🎵 Simulated Playlist:
 1. Top Hits - Todays Top Hits
 2. Top Hits - 2022 Hit Songs Music Mix
 3. Top Hits - Todays Top Hits
 4. Top Hits - Instrumental Popular Songs 2023
 5. ยิ่งดุยิ่งชอบ - Billkin
 6. ทิ้งให้จำ - Indigo
 7. Viva La Vida - Coldplay
 8. Forget Him - Bonus Track - Bobby Rydell
 9. Top Hits - Instrumental Popular Songs 2023
10. Hit 'Em Up - Single Version - 2Pac
11. Top Hits - Top TikTok Hits Now
12. Top Hits - Popular Songs
13. Top Hits - Greatest Hits 2023
14. Top Hits - Todays Top Hits
15. These Days - Rudimental
16. ยิ่งดุยิ่งชอบ - Billkin
17. เตลิด (sucker) - LUSS
18. Top Hits - Todays Top Hits
19. Top Hits - 2022 Hit Songs Music Mix
20. Top Hits - Hot Hits USA
21. ภาพที่สวยที่สุด - Ink Waruntorn
22. See You Again - Wiz Khalifa
23. Viva La Vida - Coldplay
24. Top Hits - Popular Songs
25. Forget Him - Bonus Track - Bobby Rydell
26. ฟีลลิ่งแบบว่าอู้วว! - BUS
27. Top Hits - Instrumental Popular Songs 2023
28. ฟีลลิ่งแบบว่าอู้วว! - BUS
29. Top Hits - Todays Top Hits
30.

### --------------------------------------

### Best of the Best

In [147]:
import time
import random
from collections import defaultdict

# Function to run a single method once
def run_test(method_name, cache_class, song_requests, cache_size=5):
    cache = cache_class(cache_size)
    start = time.time()
    for song in song_requests:
        cache.access_song(song)
    end = time.time()
    duration = end - start
    hits = getattr(cache, 'cache_hits', 0)
    misses = getattr(cache, 'cache_misses', len(song_requests) - hits)
    return (method_name, hits, misses, duration)

# Dummy Normal Streaming class
class NormalStreaming:
    def __init__(self, size):  # Dummy param
        self.cache_hits = 0
        self.cache_misses = 0

    def access_song(self, song):
        self.cache_misses += 1
        time.sleep(0.005)

# Overall simulation config
simulations = 100
cache_size = 5
playlist_size = 50
requests_per_simulation = 100

# Track how many times each method is the fastest
method_win_counter = defaultdict(int)

print(f"🔁 Running {simulations} simulations...\n")

for run in range(1, simulations + 1):
    # Generate random playlist
    spotify_songs = get_spotify_songs(limit=playlist_size)
    song_requests = [random.choice(spotify_songs) for _ in range(requests_per_simulation)]

    # Run all methods
    results = []
    results.append(run_test("FIFO", FIFOCache, song_requests, cache_size))
    results.append(run_test("LRU", LRUCache, song_requests, cache_size))
    results.append(run_test("LFU", LFUCache, song_requests, cache_size))
    results.append(run_test("MRU", MRUCache, song_requests, cache_size))
    results.append(run_test("Normal", NormalStreaming, song_requests, cache_size))

    # Determine winner of this run (shortest time)
    best = min(results, key=lambda x: x[3])
    method_win_counter[best[0]] += 1

    print(f"🏁 Simulation {run}: Best Method ➤ {best[0]} ({best[3]:.5f} sec)")

# Final Summary
print("\n🎯 FINAL SUMMARY AFTER 100 SIMULATIONS")
print(f"{'Method':<8} | {'Wins':<5}")
print("-" * 20)
for method, wins in sorted(method_win_counter.items(), key=lambda x: x[1], reverse=True):
    print(f"{method:<8} | {wins:<5}")

# Declare Best of the Best
best_of_best = max(method_win_counter.items(), key=lambda x: x[1])
print(f"\n🏆 BEST OF THE BEST: {best_of_best[0]} (Won {best_of_best[1]} out of {simulations} runs)")


🔁 Running 100 simulations...

🏁 Simulation 1: Best Method ➤ MRU (0.00008 sec)
🏁 Simulation 2: Best Method ➤ MRU (0.00013 sec)
🏁 Simulation 3: Best Method ➤ MRU (0.00013 sec)
🏁 Simulation 4: Best Method ➤ MRU (0.00014 sec)
🏁 Simulation 5: Best Method ➤ MRU (0.00013 sec)
🏁 Simulation 6: Best Method ➤ MRU (0.00014 sec)
🏁 Simulation 7: Best Method ➤ LRU (0.00017 sec)
🏁 Simulation 8: Best Method ➤ MRU (0.00015 sec)
🏁 Simulation 9: Best Method ➤ MRU (0.00015 sec)
🏁 Simulation 10: Best Method ➤ MRU (0.00003 sec)
🏁 Simulation 11: Best Method ➤ MRU (0.00014 sec)
🏁 Simulation 12: Best Method ➤ MRU (0.00011 sec)
🏁 Simulation 13: Best Method ➤ LRU (0.00017 sec)
🏁 Simulation 14: Best Method ➤ MRU (0.00015 sec)
🏁 Simulation 15: Best Method ➤ MRU (0.00013 sec)
🏁 Simulation 16: Best Method ➤ MRU (0.00016 sec)
🏁 Simulation 17: Best Method ➤ MRU (0.00013 sec)
🏁 Simulation 18: Best Method ➤ LRU (0.00022 sec)
🏁 Simulation 19: Best Method ➤ MRU (0.00013 sec)
🏁 Simulation 20: Best Method ➤ MRU (0.00003 sec)